# Prodigy_DecoderFix_R29 — Attaccare il rank-collapse del decoder

## Contesto

R27 ha rivelato **rank-collapse catastrofico**: 18/24 run hanno `rank_effective=1` su `Cov(decoded_params)`. Il decoder collassa a UNA SOLA dimensione effettiva su 5. R28 ha escluso Prodigy come bottleneck. R29 attacca direttamente il decoder.

**Baseline R29 = R25_A3** (`seq_len=100`, λ_T_aux=0, no other R25 tweaks). Era il top T_intra storico (0.058 vs B1=0.028).

## Due fix codice introdotti

1. **DEC-3 (init_bias_shift)**: `model.calibrate_decode_offset(first_batch)` PRIMA del training. Misura raw output medio per canale e lo sottrae → sigmoid(0)=0.5 → params al midpoint dei bound al t=0. Sostituisce il fix-#2 incompleto (row-mean subtraction post-Xavier che non considerava asimmetria spike-rate × pesi × n_ticks).

2. **DEC-1 (logit τ-annealing)**: `_decode_params(raw)` → `sigmoid((raw - offset) / τ)`. Annealing della temperatura: τ_init=5 (sigmoid "piatta", evita saturazione) → τ_final=1 (sigmoid normale, fine-tune nella zona ottimale). Schedule lineare o esponenziale. Per-channel τ supportato per scenari avanzati.

## Sweep esteso (12 esperimenti, 6 assi)

| Asse | # run | Test |
|---|---:|---|
| **E** controlli | 2 | A3 baseline + Po2-off (rank collapse causato da Po2?) |
| **A** init shift isolato | 1 | DEC-3 alone |
| **B** τ-sweep isolato | 4 | τ=3/5/10 lineare + τ=5 esponenziale |
| **C** combo init+τ | 3 | DEC-1+DEC-3 con τ=5/10 + per-channel |
| **D** longer training | 2 | C1 a 20 epoch + C2 a 20 epoch |

## Decisioni che escono da R29
- `rank_effective` passa da 1 a ≥3 in qualche setup? → fix decoder sblocca il collasso
- `T_intra_corr` > 0.10 in qualche setup? → fix decoder rompe la degenerazione
- `v0_pred`, `s0_pred` non saturano più (midpoint atteso ep1)? → init shift funziona
- Se NESSUN fix muove rank → conferma identificabilità (R30) come ultimo elemento

## Architettura

Mirror struttura `Prodigy_Tuning_R28.ipynb` (kernel-resilient validato). Cell 5 sweep idempotente + push per-run + stream stdout. Cell 6-8 standalone.

Output: `results/Prodigy_Study/DecoderFix_R29/<axis>/<tag>/`

In [ ]:
# Cell 1 -- Bootstrap + GLOBALS + ENV check
import sys, os, subprocess
import importlib.util as _imu

RESULTS_DIR = 'results/Prodigy_Study/DecoderFix_R29'
AGGREGATE_CSV = f'{RESULTS_DIR}/_aggregate_R29.csv'
BRANCH = 'Prodigy_Deep_Study'
_TMP_MSG = '/tmp/r29_msg.txt' if os.path.isdir('/tmp') else 'r29_msg.txt'
os.makedirs(RESULTS_DIR, exist_ok=True)

for pkg in ['prodigyopt', 'pandas', 'matplotlib']:
    if _imu.find_spec(pkg) is None:
        print(f'  installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for f in ['train.py', 'core/network.py',
          'results/Prodigy_Study/Audit_R27/audit_summary.csv',
          'results/Prodigy_Study/ProdigyTuning_R28/_aggregate_R28.csv',
          'results/Prodigy_Study/Ablation_R25/A_memory/R25_A3_seq_len_long/config_snapshot.json']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

sys.path.insert(0, '.')
for mod in ['train','core.network']:
    if mod in sys.modules: del sys.modules[mod]

# Verifica R29 changes attive
from core.network import CF_FSNN_Net
import torch
m = CF_FSNN_Net(hidden_size=32, rank=8, max_delay=6, bit_shift=3)
assert hasattr(m, 'decode_offset') and m.decode_offset.shape == (5,), 'R29 DEC-3 missing'
assert hasattr(m, 'logit_tau') and m.logit_tau.shape == (5,), 'R29 DEC-1 missing'
assert hasattr(m, 'set_logit_tau'), 'set_logit_tau missing'
assert hasattr(m, 'calibrate_decode_offset'), 'calibrate_decode_offset missing'
print('  [OK] core/network.py: decode_offset + logit_tau + set/calibrate methods')

# Verifica train.py CLI
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--cf_init_bias_shift', '--cf_logit_tau_init', '--cf_logit_tau_final',
             '--cf_logit_tau_schedule', '--cf_logit_tau_per_channel']:
    assert flag in help_txt, f'MISSING CLI: {flag}'
print('  [OK] CLI: 5 nuovi flag R29 disponibili')

# Verifica R27 changes presenti (val_T_intra_corr + LAYER_MAP fix)
from train import CSVLogger, BatchCSVLogger
assert 'val_T_intra_corr' in CSVLogger.COLS
print('  [OK] R27: val_T_intra_corr in CSVLogger.COLS')

br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
assert br == BRANCH, f'Branch errato: {br}'
print(f'\n[GIT] branch: {br}')
print(f'\n[GLOBALS]')
print(f'  RESULTS_DIR   = {RESULTS_DIR}')
print(f'  AGGREGATE_CSV = {AGGREGATE_CSV}')
print('\nENV check passed.')

In [ ]:
# Cell 2 -- EXPERIMENTS (12 run: 1 baseline + 1 Po2-off + 1 init + 4 tau + 3 combo + 2 long)

# Config base derivata da R25_A3 (top T_intra R27, seq_len=100, no T_aux):
COMMON = {
    'optimizer': 'prodigy', 'lr': 1.0,
    'd0': 1e-6, 'd_coef': 1.0, 'growth_rate': 'inf',
    'epochs': 10, 'max_steps_per_epoch': 100,
    'seq_len': 100,                              # A3 caratteristico (vs B1 50)
    'hidden_size': 32, 'rank': 8,
    'max_delay': 6, 'bit_shift': 3,
    'lambda_sr': 0.5, 'lambda_T_aux': 0.0,       # A3: no T_aux
    'scenario_mix': 'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',
    'cut_in_ratio': 0.0,
    'scheduler': 'cosine_no_restart',
    'T0': 5,
    'cache_path': 'data/cache_1500_mixed_cut0.0_ou0.0.pt',
    'po2_enabled': 1,                            # default ON
    # R29 DECODER FIX flags (default = no-op)
    'init_bias_shift': 0,
    'tau_init': 1.0, 'tau_final': 1.0,
    'tau_schedule': 'const',                     # const | linear | exp
    'tau_per_channel': None,                     # opt: 'v0,T,s0,a,b' csv
}

def _exp(tag, desc, axis, **overrides):
    e = {**COMMON, 'tag': tag, 'desc': desc, 'axis': axis}
    e.update(overrides)
    return e

EXPERIMENTS = [
    # ── E: controlli (baseline + Po2-off) ──
    _exp('R29_E0_baseline_A3', 'A3 replica (sanity check)',
         'E_control'),
    _exp('R29_E1_no_po2', 'A3 senza Po2 quantization (rank collapse causato da Po2?)',
         'E_control', po2_enabled=0),
    # ── A: DEC-3 init bias shift isolato ──
    _exp('R29_A1_init_shift', 'DEC-3 isolato: calibrate decode_offset',
         'A_init', init_bias_shift=1),
    # ── B: DEC-1 τ-annealing isolato (4 varianti) ──
    _exp('R29_B1_tau3to1_lin', 'τ 3→1 lineare (mild flatten)',
         'B_tau', tau_init=3.0, tau_final=1.0, tau_schedule='linear'),
    _exp('R29_B2_tau5to1_lin', 'τ 5→1 lineare (canonical anneal)',
         'B_tau', tau_init=5.0, tau_final=1.0, tau_schedule='linear'),
    _exp('R29_B3_tau10to1_lin', 'τ 10→1 lineare (strong flatten)',
         'B_tau', tau_init=10.0, tau_final=1.0, tau_schedule='linear'),
    _exp('R29_B4_tau5to1_exp', 'τ 5→1 esponenziale (geom decay)',
         'B_tau', tau_init=5.0, tau_final=1.0, tau_schedule='exp'),
    # ── C: combo DEC-3 + DEC-1 ──
    _exp('R29_C1_init_tau5', 'init_shift + τ 5→1 lineare (CANONICAL COMBO)',
         'C_combo', init_bias_shift=1, tau_init=5.0, tau_final=1.0, tau_schedule='linear'),
    _exp('R29_C2_init_tau10', 'init_shift + τ 10→1 lineare (combo aggressivo)',
         'C_combo', init_bias_shift=1, tau_init=10.0, tau_final=1.0, tau_schedule='linear'),
    _exp('R29_C3_init_per_channel', 'init_shift + τ per-channel: v0/s0 saturati = τ 10, T/a/b = τ 3',
         'C_combo', init_bias_shift=1,
         tau_per_channel='10.0,3.0,10.0,3.0,3.0',
         tau_schedule='const'),                  # per-channel statico
    # ── D: longer training di C1 e C2 (consolidamento) ──
    _exp('R29_D1_C1_epochs20', 'C1 prolungato a 20 epochs (consolidamento sblocco rank?)',
         'D_long', init_bias_shift=1, tau_init=5.0, tau_final=1.0, tau_schedule='linear',
         epochs=20),
    _exp('R29_D2_C2_epochs20', 'C2 prolungato a 20 epochs',
         'D_long', init_bias_shift=1, tau_init=10.0, tau_final=1.0, tau_schedule='linear',
         epochs=20),
]

print(f'EXPERIMENTS R29: {len(EXPERIMENTS)}')
for e in EXPERIMENTS:
    extra = ''
    if e['init_bias_shift']:    extra += ' [init_shift]'
    if e['tau_init'] != 1.0:    extra += f' [τ {e["tau_init"]}→{e["tau_final"]} {e["tau_schedule"]}]'
    if e['tau_per_channel']:    extra += f' [τ_pc={e["tau_per_channel"]}]'
    if e['po2_enabled'] == 0:   extra += ' [Po2 OFF]'
    if e['epochs'] != 10:       extra += f' [ep={e["epochs"]}]'
    print(f"  [{e['axis']:<10}] {e['tag']:<28}{extra}")
    print(f"             {e['desc']}")

In [ ]:
# Cell 3 -- Cache check (riuso da R25/R28 — mixed scenario, n_train=1500)
import os
cache = COMMON['cache_path']
if os.path.isfile(cache):
    sz_mb = os.path.getsize(cache) / 1e6
    print(f'  [OK] cache esistente: {cache}  ({sz_mb:.1f} MB)')
    print(f'       Riusata da tutti i 12 run.')
else:
    print(f'  [INFO] cache mancante: {cache}')
    print(f'         Verra\' generata dal primo run (~2-5 min).')

In [ ]:
# Cell 4 -- PRE-FLIGHT: smoke 1ep × 3step di C1 (init + τ 5→1) per validare CLI flow R29
import torch, time, shutil, sys, subprocess, json
sys.path.insert(0, '.')
if 'core.network' in sys.modules: del sys.modules['core.network']
from core.network import CF_FSNN_Net

# Static check: param count A3 setup
torch.manual_seed(42)
m = CF_FSNN_Net(hidden_size=32, rank=8, max_delay=6, bit_shift=3)
n_params = sum(p.numel() for p in m.parameters())
assert n_params == 864, f'Param count mismatch: {n_params}'
print(f'  [OK] CF_FSNN_Net A3: {n_params} param')

# Smoke: tag univoco timestamp (NFS-safe), 1ep × 3step, C1 config (init + τ 5→1)
tag_smoke = f'_R29_PREFLIGHT_{int(time.time())}'
cmd_smoke = [sys.executable, 'train.py',
    '--training_method', 'baseline',
    '--epochs', '1', '--max_steps_per_epoch', '3',
    '--batch_size', '8', '--val_batch_size', '32',
    '--seq_len', '100',
    '--cf_hidden_size', '32', '--cf_rank', '8',
    '--cf_max_delay', '6', '--cf_bit_shift', '3',
    # ← R29 flags attivi (test piu' importante: init + τ anneal)
    '--cf_init_bias_shift', '1',
    '--cf_logit_tau_init', '5.0', '--cf_logit_tau_final', '1.0',
    '--cf_logit_tau_schedule', 'linear',
    '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
    '--lambda_bc', '1.0', '--lambda_sr', '0.5', '--lambda_T_aux', '0.0',
    '--scenario_mix', COMMON['scenario_mix'],
    '--cut_in_ratio', '0.0', '--noise_scale', '0.0',
    '--n_train', '80', '--n_val', '40',
    '--optimizer', 'prodigy', '--lr', '1.0',
    '--scheduler', 'cosine_no_restart',
    '--prodigy_betas', '0.9,0.99', '--prodigy_d_coef', '1.0',
    '--prodigy_d0', '1e-6', '--prodigy_weight_decay', '0.01',
    '--prodigy_use_bias_correction', '1', '--prodigy_safeguard_warmup', '1',
    '--prodigy_growth_rate', 'inf',
    '--max_inf_streak', '99999', '--early_stop_patience', '0',
    '--tag', tag_smoke]
print(f'\n  Running smoke (init_shift=1, τ 5→1 linear, 1ep×3step)')
r = subprocess.run(cmd_smoke, capture_output=True, text=True, encoding='utf-8', errors='replace')
if r.returncode != 0:
    print('STDOUT:', r.stdout[-2000:])
    print('STDERR:', r.stderr[-500:])
    raise RuntimeError(f'preflight smoke failed (exit {r.returncode})')

# Validazioni
cfg = json.load(open(f'checkpoints/{tag_smoke}/config_snapshot.json'))
assert cfg['cf_init_bias_shift'] == 1
assert cfg['cf_logit_tau_init']  == 5.0
assert cfg['cf_logit_tau_schedule'] == 'linear'
print(f'  [OK] config_snapshot R29 flags: init_shift=1, τ 5→1 linear')

# Output stdout: cerca i messaggi DEC-3 e DEC-1
assert '[R29 DEC-3] decode_offset calibrato' in r.stdout, 'DEC-3 calibration message missing'
assert '[R29 DEC-1] logit_tau init=5.0' in r.stdout, 'DEC-1 init message missing'
print(f'  [OK] DEC-3 + DEC-1 attivi nell\'output train.py')

# Cleanup (NFS-safe)
def _robust_rmtree(path, max_retries=3):
    for attempt in range(max_retries):
        if not os.path.isdir(path): return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path): return True
        time.sleep(0.5 * (attempt + 1))
    return not os.path.isdir(path)
import os
_robust_rmtree(f'checkpoints/{tag_smoke}')
print(f'\nPRE-FLIGHT R29 passed.')

In [ ]:
# Cell 5 -- SWEEP 12 run con PUSH per-run (pattern R28: stream stdout + idempotenza)
import subprocess, sys, time, os, shutil, glob, datetime
import pandas as pd

if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/DecoderFix_R29'
    BRANCH = 'Prodigy_Deep_Study'
    _TMP_MSG = '/tmp/r29_msg.txt' if os.path.isdir('/tmp') else 'r29_msg.txt'
os.makedirs(RESULTS_DIR, exist_ok=True)

SKIP_IF_EXISTS = True

def _robust_rmtree(path, max_retries=3):
    for attempt in range(max_retries):
        if not os.path.isdir(path): return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path): return True
        time.sleep(0.5 * (attempt + 1))
    shutil.rmtree(path, ignore_errors=True)
    return not os.path.isdir(path)

def _build_cli(e):
    cli = [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', str(e['epochs']),
        '--max_steps_per_epoch', str(e['max_steps_per_epoch']),
        '--batch_size', '8', '--val_batch_size', '32',
        '--seq_len', str(e['seq_len']),
        '--cf_hidden_size', str(e['hidden_size']),
        '--cf_rank', str(e['rank']),
        '--cf_max_delay', str(e['max_delay']),
        '--cf_bit_shift', str(e['bit_shift']),
        # R29 decoder fix flags
        '--cf_init_bias_shift', str(e['init_bias_shift']),
        '--cf_logit_tau_init',  str(e['tau_init']),
        '--cf_logit_tau_final', str(e['tau_final']),
        '--cf_logit_tau_schedule', e['tau_schedule'],
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', str(e['lambda_sr']),
        '--lambda_T_aux', str(e['lambda_T_aux']),
        '--scenario_mix', e['scenario_mix'],
        '--cut_in_ratio', str(e['cut_in_ratio']),
        '--noise_scale', '0.0', '--po2_enabled', str(e['po2_enabled']),
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--data_cache', e['cache_path'],
        '--optimizer', e['optimizer'],
        '--lr', str(e['lr']), '--max_lr', str(e['lr']),
        '--scheduler', e['scheduler'],
        '--T0', str(e['T0']),
        '--prodigy_betas', '0.9,0.99',
        '--prodigy_d_coef', str(e['d_coef']),
        '--prodigy_d0', str(e['d0']),
        '--prodigy_weight_decay', '0.01',
        '--prodigy_use_bias_correction', '1',
        '--prodigy_safeguard_warmup', '1',
        '--prodigy_growth_rate', str(e['growth_rate']),
        '--tag', e['tag']]
    # per-channel tau (opzionale)
    if e['tau_per_channel']:
        cli.extend(['--cf_logit_tau_per_channel', e['tau_per_channel']])
    return cli

def _dst_for(e):
    return f"{RESULTS_DIR}/{e['axis']}/{e['tag']}"

def _push_run(e):
    tag = e['tag']
    src = f'checkpoints/{tag}'
    dst = _dst_for(e)
    if not os.path.isdir(src):
        print(f'   [WARN push] {src} mancante')
        return False
    _robust_rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')
    val_str = ''
    log_path = f'{dst}/training_log.csv'
    if os.path.isfile(log_path):
        try:
            edf = pd.read_csv(log_path)
            if len(edf) > 0:
                bi = int(edf.val_total.idxmin())
                tc  = edf.get('val_T_tracking_corr', pd.Series([float('nan')])).iloc[bi]
                tci = edf.get('val_T_intra_corr',   pd.Series([float('nan')])).iloc[bi]
                v0_e1 = edf.get('val_v0_pred_mean',  pd.Series([float('nan')])).iloc[0]
                val_str = (f'best val={edf.val_total.min():.4f}  T_corr={tc:.3f}  '
                           f'T_intra={tci:.3f}  v0_pred(ep1)={v0_e1:.1f}  '
                           f'(E{bi+1}/{len(edf)})')
        except Exception as ex:
            val_str = f'(log read failed: {ex})'
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (f"results (R29 DecoderFix): {tag} ({ts})\n\n{val_str}\n"
           f"desc={e['desc']}\nAxis: {e['axis']}\nBranch: {BRANCH}\n")
    with open(_TMP_MSG, 'w', encoding='utf-8') as fp:
        fp.write(msg)
    try:
        subprocess.run(['git','add',dst], check=True, capture_output=True)
        r = subprocess.run(['git','commit','-F',_TMP_MSG], capture_output=True, text=True)
        if r.returncode != 0:
            if 'nothing to commit' in r.stdout or 'nothing to commit' in r.stderr:
                return True
            print(f'   [push commit fail] {r.stderr[-300:]}'); return False
        subprocess.run(['git','pull','--no-rebase','--no-edit','origin',BRANCH],
                       capture_output=True, text=True)
        r2 = subprocess.run(['git','push','origin',BRANCH], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'   [push fail] {r2.stderr[-300:]}'); return False
        print(f'   [push OK]')
        return True
    finally:
        try: os.remove(_TMP_MSG)
        except: pass

if 'EXPERIMENTS' not in dir():
    raise RuntimeError('EXPERIMENTS non definito: rilancia Cell 2 prima.')

run_results = []
t_start = time.time()
total = len(EXPERIMENTS)

for i, e in enumerate(EXPERIMENTS, 1):
    tag = e['tag']
    dst = _dst_for(e)
    dst_log = f'{dst}/training_log.csv'
    if SKIP_IF_EXISTS and os.path.isfile(dst_log):
        try:
            edf = pd.read_csv(dst_log)
            v_str = f'val={edf.val_total.min():.4f} epochs={len(edf)}/{e["epochs"]}'
            if len(edf) >= e['epochs'] * 0.8:
                print(f'\n[{i}/{total}] [SKIP] {tag}: {v_str}')
                run_results.append({'tag': tag, 'axis': e['axis'], 'status':'skipped'})
                continue
            else:
                print(f'\n[{i}/{total}] [PARTIAL] {tag}: {v_str} -> RERUN')
        except Exception:
            print(f'\n[{i}/{total}] [UNREADABLE] {tag} -> RERUN')

    print(f'\n{"="*78}\n[{i}/{total}] {tag}  [axis={e["axis"]}]\n  {e["desc"]}\n{"="*78}')
    t0 = time.time()
    r = subprocess.run(_build_cli(e), capture_output=False)   # stream stdout
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    el_tot = time.time() - t_start
    done_now = sum(1 for x in run_results if x['status']!='skipped') + 1
    eta_min = (el_tot / max(done_now,1)) * (total - i) / 60
    print(f'\n[{i}/{total}] {tag} -> {status}  ({el/60:.1f}min)  ETA={eta_min:.0f}min')
    pushed = _push_run(e)
    run_results.append({'tag': tag, 'axis': e['axis'], 'status': status, 'pushed': pushed})

print(f'\n{"="*78}\nSWEEP R29 DONE in {(time.time()-t_start)/60:.0f}min')
for r in run_results:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<35} [{r['axis']:<10}] {r['status']:<10} push={r.get('pushed','N/A')}")

In [ ]:
# Cell 6 -- AGGREGATOR + comparison (R29 focus: rank_effective + T_intra + v0_pred_ep1)
import os, json, sys
import pandas as pd, numpy as np
import torch
from IPython.display import display, Markdown
sys.path.insert(0, '.')

if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/DecoderFix_R29'
    AGGREGATE_CSV = f'{RESULTS_DIR}/_aggregate_R29.csv'

# Discovery + caricamento
run_dirs = []
for root, _, files in os.walk(RESULTS_DIR):
    if 'training_log.csv' in files and 'config_snapshot.json' in files:
        run_dirs.append(root)
run_dirs = sorted(run_dirs)
print(f'Run R29 discovered: {len(run_dirs)}')

# Funzione: compute rank_effective + cond_number da best_model.pt (analogo audit R27)
def _compute_rank_for_run(run_dir, cfg):
    """Carica best_model.pt, fa forward su val set sintetico, calcola rank+cond.
    Skip se best_model.pt non trovato."""
    from core.network import CF_FSNN_Net
    tag = cfg['tag']
    ck_paths = [f'checkpoints/{tag}/best_model.pt',
                f'{run_dir}/best_model.pt']
    ck = None
    for p in ck_paths:
        if os.path.isfile(p): ck = p; break
    if ck is None:
        return np.nan, np.nan
    hs = cfg.get('cf_hidden_size', 32)
    rk = cfg.get('cf_rank', 8)
    md = cfg.get('cf_max_delay', 6)
    bs = cfg.get('cf_bit_shift', 3)
    m = CF_FSNN_Net(hidden_size=hs, rank=rk, max_delay=md, bit_shift=bs)
    sd = torch.load(ck, map_location='cpu', weights_only=False)
    m.load_state_dict(sd['model_state'])
    m.eval()
    # Forward su input sintetico uniforme (sostituto leggero per audit completo)
    torch.manual_seed(0)
    x = torch.rand(16, 50, 4)
    with torch.no_grad():
        out = m.forward_sequence(x)            # (B, T, 5)
    P = out.reshape(-1, 5).numpy()
    Pc = P - P.mean(axis=0, keepdims=True)
    cov = (Pc.T @ Pc) / max(P.shape[0] - 1, 1)
    sv = np.linalg.svd(cov, compute_uv=False)
    sv_norm = sv / max(sv[0], 1e-12)
    rank_eff = int((sv_norm > 0.01).sum())
    cond_num = float(sv[0] / max(sv[-1], 1e-12))
    return rank_eff, cond_num

rows = []
for rd in run_dirs:
    cfg = json.load(open(os.path.join(rd, 'config_snapshot.json')))
    edf = pd.read_csv(os.path.join(rd, 'training_log.csv'))
    if len(edf) == 0: continue
    bi = int(edf['val_total'].idxmin())
    # Rank/cond (best-effort: carica checkpoint se disponibile)
    try:
        rank_eff, cond_num = _compute_rank_for_run(rd, cfg)
    except Exception as ex:
        print(f'  [WARN] rank skip {cfg["tag"]}: {ex}')
        rank_eff, cond_num = np.nan, np.nan
    row = {
        'tag': cfg['tag'],
        'axis': os.path.basename(os.path.dirname(rd)),
        'n_ep': len(edf),
        'best_ep': bi+1,
        'val_total_best': float(edf['val_total'].iloc[bi]),
        'val_data_best':  float(edf['val_data'].iloc[bi]),
        'T_tracking_corr_best': float(edf.get('val_T_tracking_corr', pd.Series([np.nan])).iloc[bi]),
        'T_intra_corr_best':    float(edf.get('val_T_intra_corr',   pd.Series([np.nan])).iloc[bi]),
        # CRITICAL R29: v0_pred all'epoca 1 (test init_shift effetto)
        'v0_pred_ep1':  float(edf.get('val_v0_pred_mean', pd.Series([np.nan])).iloc[0]),
        's0_pred_ep1':  float(edf.get('val_s0_pred_mean', pd.Series([np.nan])).iloc[0]),
        'a_pred_ep1':   float(edf.get('val_a_pred_mean',  pd.Series([np.nan])).iloc[0]),
        'v0_pred_best': float(edf.get('val_v0_pred_mean', pd.Series([np.nan])).iloc[bi]),
        's0_pred_best': float(edf.get('val_s0_pred_mean', pd.Series([np.nan])).iloc[bi]),
        # Rank/Cond da checkpoint
        'rank_effective': rank_eff,
        'cond_number':    cond_num,
        # R29 config
        'init_shift':  int(cfg.get('cf_init_bias_shift', 0)),
        'tau_init':    float(cfg.get('cf_logit_tau_init', 1.0)),
        'tau_final':   float(cfg.get('cf_logit_tau_final', 1.0)),
        'tau_sched':   cfg.get('cf_logit_tau_schedule', 'const'),
        'tau_per_ch':  cfg.get('cf_logit_tau_per_channel', None),
        'po2_enabled': int(cfg.get('po2_enabled', 1)),
    }
    rows.append(row)

df = pd.DataFrame(rows).sort_values('T_intra_corr_best', ascending=False)
df.to_csv(AGGREGATE_CSV, index=False)
print(f'Saved: {AGGREGATE_CSV}')

display(Markdown('## R29 sintesi — ordinata per T_intra_corr desc'))
show = ['tag','axis','init_shift','tau_init','tau_final','tau_sched',
        'val_data_best','T_tracking_corr_best','T_intra_corr_best',
        'rank_effective','cond_number','v0_pred_ep1','s0_pred_ep1']
show = [c for c in show if c in df.columns]
display(df[show].round(4))

# ── Confronto vs baseline R29_E0 ──
if 'R29_E0_baseline_A3' in df['tag'].values:
    base = df[df.tag=='R29_E0_baseline_A3'].iloc[0]
    display(Markdown(f'## Delta vs R29_E0 (Δ = run − baseline A3)'))
    delta = []
    for _, r in df.iterrows():
        if r['tag'] == 'R29_E0_baseline_A3': continue
        delta.append({
            'tag': r['tag'],
            'Δ val_data':  r['val_data_best']  - base['val_data_best'],
            'Δ T_intra':   r['T_intra_corr_best']    - base['T_intra_corr_best'],
            'Δ rank':      r['rank_effective']  - base['rank_effective'] if not np.isnan(base['rank_effective']) else np.nan,
            'Δ v0_pred_ep1': r['v0_pred_ep1']  - base['v0_pred_ep1'],
        })
    display(pd.DataFrame(delta).round(4))

# ── Verdetto per asse ──
display(Markdown('## Verdetti per asse'))
for axis in ['E_control','A_init','B_tau','C_combo','D_long']:
    sub = df[df.axis == axis]
    if sub.empty: continue
    print(f'\n[{axis}]')
    for _, r in sub.iterrows():
        rank_ok    = '✅' if r['rank_effective'] >= 3 else ('⚠' if r['rank_effective'] >= 2 else '❌')
        intra_ok   = '✅' if r['T_intra_corr_best'] > 0.10 else ('⚠' if r['T_intra_corr_best'] > 0.058 else '❌')
        init_ok    = '✅' if abs(r['v0_pred_ep1'] - 26.5) < 5 else ('⚠' if r['v0_pred_ep1'] < 38 else '❌')  # midpoint 26.5, alert se >38 = saturato
        print(f'  {r["tag"]:<32}  rank={r["rank_effective"]!s:<3} [{rank_ok}]  '
              f'T_intra={r["T_intra_corr_best"]:.3f} [{intra_ok}]  '
              f'v0_pred_ep1={r["v0_pred_ep1"]:.1f} [{init_ok}]')

In [ ]:
# Cell 7 -- PLOT suite (decoder behavior + rank + T_intra evolution)
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt

if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/DecoderFix_R29'
if 'df' not in dir():
    df = pd.read_csv(f'{RESULTS_DIR}/_aggregate_R29.csv')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: v0_pred ep1 vs ep_best (init_shift effetto?)
ax = axes[0,0]
x = df['v0_pred_ep1']
y = df['v0_pred_best']
ax.scatter(x, y, s=120, alpha=0.7, edgecolor='black',
           c=df['init_shift'], cmap='RdYlGn', vmin=0, vmax=1)
ax.axhline(26.5, color='black', linestyle=':', alpha=0.5, label='v0 midpoint=26.5')
ax.axvline(26.5, color='black', linestyle=':', alpha=0.5)
ax.axhline(45, color='red', linestyle=':', alpha=0.3, label='v0 MAX=45 (saturazione)')
ax.axvline(45, color='red', linestyle=':', alpha=0.3)
for _, r in df.iterrows():
    ax.annotate(r['tag'].replace('R29_','')[:10], (r['v0_pred_ep1'], r['v0_pred_best']),
                fontsize=8, alpha=0.8)
ax.set_xlabel('v0_pred ep1 (init bias shift effect)')
ax.set_ylabel('v0_pred best epoch')
ax.set_title('init_bias_shift: v0 saturation broken? (verde = shift attivo)')
ax.legend(); ax.grid(alpha=0.3)

# Plot 2: rank_effective per run
ax = axes[0,1]
df_sorted = df.sort_values('rank_effective', ascending=True)
colors = ['red' if r < 2 else ('orange' if r < 3 else 'green') for r in df_sorted['rank_effective']]
ax.barh(range(len(df_sorted)), df_sorted['rank_effective'], color=colors, alpha=0.7)
ax.set_yticks(range(len(df_sorted)))
ax.set_yticklabels([t.replace('R29_','') for t in df_sorted['tag']], fontsize=8)
ax.axvline(1, color='gray', linestyle=':', alpha=0.5, label='rank=1 (collapse R27)')
ax.axvline(3, color='green', linestyle=':', alpha=0.5, label='rank=3 (sblocco atteso)')
ax.axvline(5, color='blue', linestyle=':', alpha=0.5, label='rank=5 (full)')
ax.set_xlabel('rank_effective')
ax.set_title('R29: Sblocco del rank-collapse?')
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='x')

# Plot 3: T_intra evolution per epoca (curve)
ax = axes[1,0]
for _, r in df.iterrows():
    log_path = f'{RESULTS_DIR}/{r["axis"]}/{r["tag"]}/training_log.csv'
    if not os.path.isfile(log_path): continue
    edf = pd.read_csv(log_path)
    if 'val_T_intra_corr' not in edf.columns: continue
    label = r['tag'].replace('R29_','')[:18]
    style = '-' if r['init_shift'] == 1 else '--'
    ax.plot(edf['epoch'], edf['val_T_intra_corr'], marker='o', label=label, linestyle=style, alpha=0.8)
ax.set_xlabel('epoch'); ax.set_ylabel('val_T_intra_corr')
ax.axhline(0.058, color='gray', linestyle=':', alpha=0.5, label='R27 top historic (A3)')
ax.axhline(0.10, color='orange', linestyle=':', alpha=0.5, label='soglia weak unlock')
ax.axhline(0.30, color='green', linestyle=':', alpha=0.5, label='soglia strong')
ax.set_title('T_intra_corr evolution (linee solide = init_shift attivo)')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Plot 4: Pareto val_data vs T_intra
ax = axes[1,1]
ax.scatter(df['val_data_best'], df['T_intra_corr_best'], s=120, alpha=0.7,
           c=df['rank_effective'], cmap='viridis', edgecolor='black')
cbar = plt.colorbar(ax.collections[0], ax=ax, label='rank_effective')
for _, r in df.iterrows():
    ax.annotate(r['tag'].replace('R29_','')[:10],
                (r['val_data_best'], r['T_intra_corr_best']),
                fontsize=8, alpha=0.8)
ax.set_xlabel('val_data_best (lower better)')
ax.set_ylabel('T_intra_corr_best (higher better)')
ax.set_title('R29 Pareto: val_data ↔ T_intra (colore = rank)')
ax.grid(alpha=0.3)

plt.tight_layout()
out_png = f'{RESULTS_DIR}/R29_diagnostic_plots.png'
plt.savefig(out_png, dpi=110)
plt.show()
print(f'Salvato: {out_png}')

In [ ]:
# Cell 8 -- Summary + decisione passaggio R30 + push finale aggregator
import os, subprocess, tempfile, pandas as pd, numpy as np
from IPython.display import display, Markdown

if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/DecoderFix_R29'
    AGGREGATE_CSV = f'{RESULTS_DIR}/_aggregate_R29.csv'
    BRANCH = 'Prodigy_Deep_Study'
if 'df' not in dir():
    df = pd.read_csv(AGGREGATE_CSV)

display(Markdown('## Verdetto R29: il fix decoder rompe il rank-collapse?'))

any_rank_unlock = (df['rank_effective'] >= 3).any()
any_intra_strong = (df['T_intra_corr_best'] > 0.10).any()
any_intra_2x = (df['T_intra_corr_best'] > 0.058).any()  # > top R27 historic A3
any_init_fix = (df['v0_pred_ep1'] < 33).any()  # midpoint = 26.5, < 33 = non saturato

best_intra = df.sort_values('T_intra_corr_best', ascending=False).iloc[0]
best_val   = df.sort_values('val_data_best', ascending=True).iloc[0]
best_rank  = df.sort_values('rank_effective', ascending=False).iloc[0]

print(f'Run con rank_effective ≥ 3:        {any_rank_unlock}')
print(f'Run con T_intra > 0.10:           {any_intra_strong}')
print(f'Run con T_intra > 0.058 (top R27):{any_intra_2x}')
print(f'Run con v0_pred_ep1 < 33 (no sat):{any_init_fix}')
print()
print(f'Best T_intra:   {best_intra["tag"]} = {best_intra["T_intra_corr_best"]:.4f}')
print(f'Best val_data:  {best_val["tag"]} = {best_val["val_data_best"]:.4f}')
print(f'Best rank_eff:  {best_rank["tag"]} = {best_rank["rank_effective"]}/5')

if any_rank_unlock and any_intra_strong:
    display(Markdown('### ✅ Decoder fix HA rotto il rank-collapse. R29 trionfo. Procedere a validazione cross-scenario o deploy.'))
elif any_intra_2x and any_init_fix:
    display(Markdown('### ⚠ Parziale: T_intra migliorato + init shift effective ma rank ancora ≤ 2. Continuare a R30 (identifiability via supervisione esplicita) per romper rank.'))
elif any_init_fix and not any_intra_2x:
    display(Markdown('### ⚠ Init shift funziona (v0 desaturato) ma T_intra non migliora. Conferma: il problema NON è saturation, è identifiability. R30 critico.'))
else:
    display(Markdown('### ❌ Nessun fix decoder muove le metriche. Bottleneck rimane identifiability strutturale. R30 obbligatorio.'))

# Push aggregator + plots (idempotente)
subprocess.run(['git','add', AGGREGATE_CSV, f'{RESULTS_DIR}/R29_diagnostic_plots.png'], check=False)
msg = f"R29 DecoderFix aggregator: {len(df)} run, best T_intra={best_intra['T_intra_corr_best']:.3f}, best rank={best_rank['rank_effective']}"
with tempfile.NamedTemporaryFile('w', delete=False, suffix='.txt', encoding='utf-8') as fp:
    fp.write(msg); msg_path = fp.name
r = subprocess.run(['git','diff','--cached','--name-only'], capture_output=True, text=True)
if r.stdout.strip():
    subprocess.run(['git','commit','-F',msg_path], check=True)
    subprocess.run(['git','pull','--no-rebase','--no-edit','origin',BRANCH], check=True)
    subprocess.run(['git','push','origin',BRANCH], check=True)
    print('\n[OK] aggregator R29 pushato.')
else:
    print('\n[SKIP] nessuna modifica aggregator da pushare.')
try: os.unlink(msg_path)
except: pass